In [1]:
from pathlib import Path
from IPython.display import display
import gc
import pandas as pd

raw_dir = Path("../data/raw")
file_names = [
    "2019-oct.csv",
    "2019-nov.csv",
    "2019-dec.csv",
    "2020-jan.csv",
    "2020-feb.csv",
]

for file_name in file_names:
    file_path = raw_dir / file_name
    print(file_name, "exists:", file_path.exists())
    if file_path.exists():
        print("file size (MB):", round(file_path.stat().st_size / (1024 ** 2), 2))

2019-oct.csv exists: True
file size (MB): 460.19
2019-nov.csv exists: True
file size (MB): 520.55
2019-dec.csv exists: True
file size (MB): 396.06
2020-jan.csv exists: True
file size (MB): 478.55
2020-feb.csv exists: True
file size (MB): 466.16


In [2]:
sample_path = raw_dir / file_names[0]
sample_df = pd.read_csv(sample_path, nrows=5)

print(sample_df.shape)
print(sample_df.dtypes)
print(sample_df.head())

(5, 9)
event_time           str
event_type           str
product_id         int64
category_id        int64
category_code    float64
brand                str
price            float64
user_id            int64
user_session         str
dtype: object
                event_time event_type  product_id          category_id  \
0  2019-10-01 00:00:00 UTC       cart     5773203  1487580005134238553   
1  2019-10-01 00:00:03 UTC       cart     5773353  1487580005134238553   
2  2019-10-01 00:00:07 UTC       cart     5881589  2151191071051219817   
3  2019-10-01 00:00:07 UTC       cart     5723490  1487580005134238553   
4  2019-10-01 00:00:15 UTC       cart     5881449  1487580013522845895   

   category_code   brand  price    user_id  \
0            NaN  runail   2.62  463240011   
1            NaN  runail   2.62  463240011   
2            NaN  lovely  13.48  429681830   
3            NaN  runail   2.62  463240011   
4            NaN  lovely   0.56  429681830   

                           user_

In [3]:
monthly_rows = []
missing_rows = []
event_rows = []

for file_name in file_names:
    file_path = raw_dir / file_name
    print("검사 시작:", file_name)

    df = pd.read_csv(file_path)
    row_count = len(df)

    # 날짜가 날짜 형식으로 읽히는지 점검 (원본 컬럼은 변경하지 않음)
    parsed_time = pd.to_datetime(
        df["event_time"],
        utc=True,
        errors="coerce"
    )

    monthly_rows.append({
        "file_name": file_name,
        "row_count": row_count,
        "column_count": len(df.columns),
        "first_event": parsed_time.min(),
        "last_event": parsed_time.max(),
        "invalid_time_count": int(parsed_time.isna().sum()),
        "duplicate_candidate_count": int(df.duplicated().sum()),
        "price_min": df["price"].min(),
        "price_max": df["price"].max(),
        "price_zero_count": int((df["price"] == 0).sum()),
        "price_negative_count": int((df["price"] < 0).sum()),
        "unique_user_count": df["user_id"].nunique(),
        "unique_session_count": df["user_session"].nunique(),
    })

    missing_count = df.isna().sum()
    for column_name in df.columns:
        missing_rows.append({
            "file_name": file_name,
            "column_name": column_name,
            "row_count": row_count,
            "missing_count": int(missing_count[column_name]),
            "missing_rate": missing_count[column_name] / row_count,
        })

    event_counts = df["event_type"].value_counts(dropna=False)
    for event_name, event_count in event_counts.items():
        event_rows.append({
            "file_name": file_name,
            "event_type": event_name,
            "event_count": int(event_count),
            "event_rate": event_count / row_count,
        })

    del df
    del parsed_time
    gc.collect()
    print("검사 완료:", file_name)

monthly_df = pd.DataFrame(monthly_rows)
missing_df = pd.DataFrame(missing_rows)
event_df = pd.DataFrame(event_rows)

검사 시작: 2019-oct.csv
검사 완료: 2019-oct.csv
검사 시작: 2019-nov.csv
검사 완료: 2019-nov.csv
검사 시작: 2019-dec.csv
검사 완료: 2019-dec.csv
검사 시작: 2020-jan.csv
검사 완료: 2020-jan.csv
검사 시작: 2020-feb.csv
검사 완료: 2020-feb.csv


In [4]:
display(
    print(monthly_df.to_string(index=False))
)

print("5개월 전체 이벤트 행 수:", monthly_df["row_count"].sum())
print("중복 후보 수 합계(각 월 파일 내부 기준):", monthly_df["duplicate_candidate_count"].sum())
print("가격 0 이벤트 수:", monthly_df["price_zero_count"].sum())
print("음수 가격 이벤트 수:", monthly_df["price_negative_count"].sum())
print("날짜 파싱 실패 수:", monthly_df["invalid_time_count"].sum())

   file_name  row_count  column_count               first_event                last_event  invalid_time_count  duplicate_candidate_count  price_min  price_max  price_zero_count  price_negative_count  unique_user_count  unique_session_count
2019-oct.csv    4102283             9 2019-10-01 00:00:00+00:00 2019-10-31 23:59:54+00:00                   0                     213155     -79.37     307.60              6066                    20             399664                873960
2019-nov.csv    4635837             9 2019-11-01 00:00:02+00:00 2019-11-30 23:59:58+00:00                   0                     246693     -79.37     327.78             11625                    17             368232                942022
2019-dec.csv    3533286             9 2019-12-01 00:00:00+00:00 2019-12-31 23:59:57+00:00                   0                     183860     -79.37     327.78              7589                    18             370154                839812
2020-jan.csv    4264752             9 20

None

5개월 전체 이벤트 행 수: 20692840
중복 후보 수 합계(각 월 파일 내부 기준): 1109098
가격 0 이벤트 수: 104157
음수 가격 이벤트 수: 131
날짜 파싱 실패 수: 0


In [5]:
missing_summary = (
    missing_df
    .groupby("column_name", as_index=False)[["row_count", "missing_count"]]
    .sum()
)

missing_summary["missing_rate"] = (
    missing_summary["missing_count"]
    / missing_summary["row_count"]
)

display(
    print(missing_summary.to_string(index=False))
)

print("월별 결측률:")
print(missing_df.to_string(index=False))

  column_name  row_count  missing_count  missing_rate
        brand   20692840        8757117      0.423196
category_code   20692840       20339246      0.982912
  category_id   20692840              0      0.000000
   event_time   20692840              0      0.000000
   event_type   20692840              0      0.000000
        price   20692840              0      0.000000
   product_id   20692840              0      0.000000
      user_id   20692840              0      0.000000
 user_session   20692840           4598      0.000222


None

월별 결측률:
   file_name   column_name  row_count  missing_count  missing_rate
2019-oct.csv    event_time    4102283              0      0.000000
2019-oct.csv    event_type    4102283              0      0.000000
2019-oct.csv    product_id    4102283              0      0.000000
2019-oct.csv   category_id    4102283              0      0.000000
2019-oct.csv category_code    4102283        4034806      0.983551
2019-oct.csv         brand    4102283        1659261      0.404473
2019-oct.csv         price    4102283              0      0.000000
2019-oct.csv       user_id    4102283              0      0.000000
2019-oct.csv  user_session    4102283            637      0.000155
2019-nov.csv    event_time    4635837              0      0.000000
2019-nov.csv    event_type    4635837              0      0.000000
2019-nov.csv    product_id    4635837              0      0.000000
2019-nov.csv   category_id    4635837              0      0.000000
2019-nov.csv category_code    4635837        4560089  

In [6]:
event_summary = (
    event_df
    .groupby("event_type", as_index=False)["event_count"]
    .sum()
)
event_summary["event_rate"] = (
    event_summary["event_count"]
    / event_summary["event_count"].sum()
)

display(
    print(event_summary.to_string(index=False))
)

print("월별 이벤트 분포:")
print(event_df.to_string(index=False))

      event_type  event_count  event_rate
            cart      5768333    0.278760
        purchase      1287007    0.062196
remove_from_cart      3979679    0.192322
            view      9657821    0.466723


None

월별 이벤트 분포:
   file_name       event_type  event_count  event_rate
2019-oct.csv             view      1862164    0.453934
2019-oct.csv             cart      1232385    0.300414
2019-oct.csv remove_from_cart       762110    0.185777
2019-oct.csv         purchase       245624    0.059875
2019-nov.csv             view      2076132    0.447844
2019-nov.csv             cart      1311807    0.282971
2019-nov.csv remove_from_cart       925481    0.199636
2019-nov.csv         purchase       322417    0.069549
2019-dec.csv             view      1728331    0.489157
2019-dec.csv             cart       927124    0.262397
2019-dec.csv remove_from_cart       664655    0.188112
2019-dec.csv         purchase       213176    0.060334
2020-jan.csv             view      2037608    0.477779
2020-jan.csv             cart      1148323    0.269259
2020-jan.csv remove_from_cart       815024    0.191107
2020-jan.csv         purchase       263797    0.061855
2020-feb.csv             view      1953586    0.469987

## TASK-001 추가 검증

### 1. 이벤트 종류별 중복 후보 분석

목적:
- 중복 후보가 특정 이벤트 종류에 집중되는지 확인
- 중복 후보 비율을 계산하여 데이터 품질에 미치는 영향 평가
- 실제 데이터 삭제 여부를 결정하기 위한 근거 확보

In [7]:
duplicate_rows = []

for file_name in file_names:
    df = pd.read_csv(
        raw_dir / file_name
    )

    duplicate_mask = df.duplicated()

    event_count = df[
        "event_type"
    ].value_counts()

    duplicate_count = df.loc[
        duplicate_mask,
        "event_type"
    ].value_counts()

    for event_type in event_count.index:
        total = event_count[event_type]
        duplicates = duplicate_count.get(
            event_type, 0
        )

        duplicate_rows.append({
            "file_name": file_name,
            "event_type": event_type,
            "event_count": total,
            "duplicate_count": duplicates,
            "duplicate_rate": duplicates / total
        })

    del df

duplicate_detail_df = pd.DataFrame(
    duplicate_rows
)

display(
    duplicate_detail_df
)

duplicate_summary = (
    duplicate_detail_df
    .groupby(
        "event_type",
        as_index=False
    )[
        [
            "event_count",
            "duplicate_count"
        ]
    ]
    .sum()
)

duplicate_summary["duplicate_rate"] = (
    duplicate_summary["duplicate_count"]
    / duplicate_summary["event_count"]
)

display(
    duplicate_summary.style.format({
        "event_count": "{:,.0f}",
        "duplicate_count": "{:,.0f}",
        "duplicate_rate": "{:.2%}"
    })
)

duplicate_validation = (
    duplicate_detail_df
    .groupby(
        "file_name",
        as_index=False
    )["duplicate_count"]
    .sum()
)

duplicate_validation = (
    duplicate_validation
    .merge(
        monthly_df[
            [
                "file_name",
                "duplicate_candidate_count"
            ]
        ],
        on="file_name"
    )
)

duplicate_validation["is_valid"] = (
    duplicate_validation["duplicate_count"]
    == duplicate_validation["duplicate_candidate_count"]
)

print(duplicate_validation)

assert duplicate_validation["is_valid"].all()

,file_name,event_type,event_count,duplicate_count,duplicate_rate
0,2019-oct.csv,view,1862164,114,0.000061
1,2019-oct.csv,cart,1232385,28167,0.022856
2,2019-oct.csv,remove_from_cart,762110,184531,0.242132
3,2019-oct.csv,purchase,245624,343,0.001396
4,2019-nov.csv,view,2076132,139,0.000067
5,2019-nov.csv,cart,1311807,22175,0.016904
6,2019-nov.csv,remove_from_cart,925481,224231,0.242286
7,2019-nov.csv,purchase,322417,148,0.000459
8,2019-dec.csv,view,1728331,110,0.000064
9,2019-dec.csv,cart,927124,15483,0.016700


,event_type,event_count,duplicate_count,duplicate_rate
0,cart,"5,768,333","115,342",2.00%
1,purchase,"1,287,007",905,0.07%
2,remove_from_cart,"3,979,679","991,820",24.92%
3,view,"9,657,821","1,031",0.01%


      file_name  duplicate_count  duplicate_candidate_count  is_valid
0  2019-dec.csv           183860                     183860      True
1  2019-nov.csv           246693                     246693      True
2  2019-oct.csv           213155                     213155      True
3  2020-feb.csv           240290                     240290      True
4  2020-jan.csv           225100                     225100      True


### 2. 가격 이상값 분석

목적:
- 가격이 0 이하인 이벤트의 종류 확인
- 상품 및 이벤트 유형과의 관계 검토
- 향후 금액 기반 KPI 계산 시 처리 기준 검토

In [8]:
price_rows = []
session_rows = []

for file_name in file_names:
    df = pd.read_csv(
        raw_dir / file_name
    )

    # 가격 이상값 검사
    for event_type in df["event_type"].unique():
        event_df_temp = df[
            df["event_type"] == event_type
        ]

        price_rows.append({
            "file_name": file_name,
            "event_type": event_type,
            "zero_count": (
                event_df_temp["price"] == 0
            ).sum(),
            "negative_count": (
                event_df_temp["price"] < 0
            ).sum()
        })

    # 음수 가격 고유 상품 수
    negative_product_count = df.loc[
        df["price"] < 0,
        "product_id"
    ].nunique()

    # 세션별 사용자 수
    session_user_count = (
        df.dropna(
            subset=["user_session"]
        )
        .groupby("user_session")["user_id"]
        .nunique()
    )

    session_rows.append({
        "file_name": file_name,
        "negative_product_count": negative_product_count,
        "multi_user_session_count": (
            session_user_count > 1
        ).sum()
    })

    del df
    gc.collect()

price_detail_df = pd.DataFrame(
    price_rows
)

session_detail_df = pd.DataFrame(
    session_rows
)

price_summary = (
    price_detail_df
    .groupby(
        "file_name",
        as_index=False
    )[
        [
            "zero_count",
            "negative_count"
        ]
    ]
    .sum()
)

display(
    price_summary
)

display(
    session_detail_df
)

,file_name,zero_count,negative_count
0,2019-dec.csv,7589,18
1,2019-nov.csv,11625,17
2,2019-oct.csv,6066,20
3,2020-feb.csv,71254,37
4,2020-jan.csv,7623,39


,file_name,negative_product_count,multi_user_session_count
0,2019-oct.csv,5,22
1,2019-nov.csv,5,39
2,2019-dec.csv,5,42
3,2020-jan.csv,5,86
4,2020-feb.csv,5,80


In [9]:
assert (
    price_summary["zero_count"].sum()
    == monthly_df["price_zero_count"].sum()
)

assert (
    price_summary["negative_count"].sum()
    == monthly_df["price_negative_count"].sum()
)

print("가격 이상값 집계 검증 통과")

가격 이상값 집계 검증 통과


In [10]:
monthly_df[
    [
        "file_name",
        "row_count",
        "unique_user_count",
        "unique_session_count"
    ]
]

,file_name,row_count,unique_user_count,unique_session_count
0,2019-oct.csv,4102283,399664,873960
1,2019-nov.csv,4635837,368232,942022
2,2019-dec.csv,3533286,370154,839812
3,2020-jan.csv,4264752,410073,965351
4,2020-feb.csv,4156682,391055,931668
